In [48]:
!pip install stanza

#Library--------------------------------

import re
from collections import Counter
import nltk
import pandas as pd
import stanza
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required tokenization maps and dictionaries
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Download the Marathi model for Stanza
print("Downloading Stanza language files for Marathi...")
stanza.download('mr', processors='tokenize,lemma,pos')

# Initialize the background pipeline engine
nlp_marathi = stanza.Pipeline('mr', processors='tokenize,lemma,pos', use_gpu=False, quiet=True)

# Initialize Porter Stemmer
porter_stemmer = PorterStemmer()

print("✨ Environment successfully configured and ready!")

INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: mr (Marathi)...
| Processor       | Package       |
-----------------------------------
| tokenize        | ufal          |
| mwt             | ufal          |
| pos             | ufal_charlm   |
| lemma           | ufal_nocharlm |
| pretrain        | fasttextwiki  |
| forward_charlm  | l3cube        |
| backward_charlm | l3cube        |

INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/mr/tokenize/ufal.pt
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/mr/mwt/ufal.pt
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/mr/pos/ufal_charlm.pt
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/mr/lemma/ufal_nocharlm.pt
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/mr/pretrain/fasttextwiki.pt
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/mr/forward_charlm/l3cube.pt
I

INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Loading these models for language: mr (Marathi):
| Processor | Package       |
-----------------------------
| tokenize  | ufal          |
| mwt       | ufal          |
| pos       | ufal_charlm   |
| lemma     | ufal_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


✨ Environment successfully configured and ready!


In [49]:
# 10-Sentence Historical Corpus-----------------------


documents = {
    "English": ("Shaniwar Wada is a magnificent historical fortification located in the city of Pune. "
        "It was constructed in 1732 by Peshwa Baji Rao I as the main seat of the Peshwas. "
        "This palace became the political heart and grand capital of the expanding Maratha Empire. "
        "The architecture displays a brilliant blend of Maratha style and exquisite Mughal design elements. "
        "Its massive foundations consist of strong stone layers while the upper floors were built of timber. "
        "The complex feature five grand gateways including the famous Delhi Darwaza facing north. "
        "A spectacular seven-storied structure once stood proudly within the inner courtyard walls. "
        "Unfortunately an unexpected and devastating fire destroyed the entire palace complex in 1828. "
        "Today the surviving stone ruins attract thousands of tourists from all over the world. "
        "It stands as a timeless symbol of Maharashtra's rich historical heritage and pride."),


    "Hindi": ("शनिवार वाड़ा पुणे शहर में स्थित एक भव्य ऐतिहासिक किला है। "
        "इसका निर्माण 1732 में पेशवा बाजीराव प्रथम ने मुख्य निवास के रूप में कराया था। "
        "यह महल विशाल मराठा साम्राज्य का प्रमुख राजनीतिक और प्रशासनिक केंद्र बन गया। "
        "इसकी वास्तुकला में मराठा शैली और उत्तम मुगल डिजाइन का सुंदर मिश्रण दिखाई देता है। "
        "किले की मजबूत नींव पत्थरों से बनी है जबकि ऊपरी मंजिलों में लकड़ी का उपयोग हुआ था। "
        "इस परिसर में पांच बड़े मुख्य द्वार हैं जिसमें प्रसिद्ध दिल्ली दरवाजा उत्तर की ओर है। "
        "एक समय इस आंतरिक प्रांगण के भीतर एक शानदार सात मंजिला इमारत खड़ी थी। "
        "दुर्भाग्य से 1828 में एक विनाशकारी आग ने पूरे महल परिसर को नष्ट कर दिया। "
        "आज इसके बचे हुए ऐतिहासिक अवशेष दुनिया भर के हजारों पर्यटकों को आकर्षित करते हैं। "
        "यह किला महाराष्ट्र के समृद्ध इतिहास और गौरव का एक कालातीत प्रतीक माना जाता है। "),


    "Marathi": ("शनिवार वाडा हा पुणे शहरातील एक अत्यंत भव्य आणि ऐतिहासिक किल्ला आहे. "
        "या वास्तूची उभारणी 1732 मध्ये पेशवे बाजीराव पहिले यांनी मुख्य निवासस्थान म्हणून केली. "
        "हा राजवाडा बलाढ्य मराठा साम्राज्याचे प्रमुख राजकीय आणि प्रशासकीय केंद्र बनला. "
        "याच्या वास्तुकलेमध्ये मराठा साम्राज्य शैली आणि उत्कृष्ट मुघल डिझाइनचे सुंदर मिश्रण दिसून येते. "
        "किल्ल्याचा पाया मजबूत दगडांचा बनवला असून वरील मजले लाकडाचे बांधण्यात आले होते. "
        "या ऐतिहासिक वास्तूला एकूण पाच मोठे दरवाजे असून प्रसिद्ध दिल्ली दरवाजा उत्तरेकडे तोंड करून आहे. "
        "एकदा या अंतर्गत प्रांगणात एक भव्य आणि देखणी सात मजली इमारत उभी होती. "
        "दुर्दैवाने 1828 मध्ये लागलेल्या एका भीषण आगीत संपूर्ण राजवाडा परिसर नष्ट झाला. "
        "आज या किल्ल्याचे अवशेष जगभरातील हजारो पर्यटकांना पुण्याकडे आकर्षित करतात. "
        "हा वाडा महाराष्ट्राच्या समृद्ध ऐतिहासिक वारशाचे आणि अस्मितेचे एक जिवंत प्रतीक आहे.")
}



# Stopwords filters to remove common grammatical markers
en_stopwords = set(stopwords.words('english')) | {"is", "a", "by", "of", "the", "in", "was", "as", "this", "and", "its", "were", "from", "once"}
hi_stopwords = {"में", "एक", "है", "जिसका", "ने", "कराया", "था", "के", "रूप", "का", "और", "की", "इसकी", "से", "बनी", "जबकि", "हुआ", "था।", "इस", "हैं", "जिसमें", "ओर", "भीतर", "खड़ी", "थी।", "दिया", "इसके", "हुए", "को", "जाता", "माना", "है।"}
mr_stopwords = {"हा", "एक", "आहे", "जो", "च्या", "पुण्यातील", "मराठा", "या", "मध्ये", "म्हणून", "केली", "झाला", "याच्या", "येते", "चा", "असून", "होते", "एकूण", "करून", "होती", "लागलेल्या", "एका", "याने", "राजवाडा", "वाडा"}

print("✅ Data corpus and stopwords initialized.")


✅ Data corpus and stopwords initialized.


In [50]:
def hindi_stemmer(word):
    """
    Standard rule-based Hindi Suffix Stripper for structural Information Retrieval.
    Slices off common noun/verb inflectional suffixes to isolate core word bases.
    """
    # Expanded list of Hindi suffixes, ordered from longest to shortest for accurate stripping
    suffixes = [
        'ाओं', 'ाएं', 'ाइयों', 'ाएँ', 'ियों', 'ओंग', 'ावर', 'ाव', 'ान', 'ाश', 'कर',
        'ाओ', 'िए', 'ाई', 'ाँ', 'ो', 'े', 'ी', 'ा',  # Original suffixes
        'करो', 'वाला', 'वाली', 'वाले', 'पन', 'ता', 'ने', 'ना', 'नी', 'से', 'को',
        'का', 'की', 'के', 'में', 'पर', 'गा', 'गी', 'गे', 'या', 'यी', 'ये',
        'ा', 'ीं', 'ुं', 'ों', 'याँ', 'ियाँ', 'ए', 'ओ', 'उ', 'ए', 'ई', 'आ'
    ]
    for suffix in sorted(suffixes, key=len, reverse=True): # Sort to prioritize longer suffixes
        if word.endswith(suffix):
            # Maintain a minimum base length constraint so short words don't break
            if len(word) - len(suffix) >= 2:
                return word[:-len(suffix)]
    return word

print("⚙️ Hindi stemming algorithm loaded.")

⚙️ Hindi stemming algorithm loaded.


In [51]:
print('Original Hindi filtered tokens:')
print(hi_filtered)
print('\nStemmed Hindi tokens:')
print(hi_stemmed_tokens)

print('\nCleaned Hindi string:')
print(hi_clean_string)

Original Hindi filtered tokens:
['शनिवार', 'वाड़ा', 'पुणे', 'शहर', 'स्थित', 'भव्य', 'ऐतिहासिक', 'किला', 'इसका', 'निर्माण', '1732', 'पेशवा', 'बाजीराव', 'प्रथम', 'मुख्य', 'निवास', 'यह', 'महल', 'विशाल', 'मराठा', 'साम्राज्य', 'प्रमुख', 'राजनीतिक', 'प्रशासनिक', 'केंद्र', 'बन', 'गया', 'वास्तुकला', 'मराठा', 'शैली', 'उत्तम', 'मुगल', 'डिजाइन', 'सुंदर', 'मिश्रण', 'दिखाई', 'देता', 'किले', 'मजबूत', 'नींव', 'पत्थरों', 'ऊपरी', 'मंजिलों', 'लकड़ी', 'उपयोग', 'परिसर', 'पांच', 'बड़े', 'मुख्य', 'द्वार', 'प्रसिद्ध', 'दिल्ली', 'दरवाजा', 'उत्तर', 'समय', 'आंतरिक', 'प्रांगण', 'शानदार', 'सात', 'मंजिला', 'इमारत', 'थी', 'दुर्भाग्य', '1828', 'विनाशकारी', 'आग', 'पूरे', 'महल', 'परिसर', 'नष्ट', 'कर', 'आज', 'बचे', 'ऐतिहासिक', 'अवशेष', 'दुनिया', 'भर', 'हजारों', 'पर्यटकों', 'आकर्षित', 'करते', 'यह', 'किला', 'महाराष्ट्र', 'समृद्ध', 'इतिहास', 'गौरव', 'कालातीत', 'प्रतीक']

Stemmed Hindi tokens:
['शनिवार', 'वाड़', 'पुण', 'शहर', 'स्थित', 'भव्य', 'ऐतिहासिक', 'किल', 'इस', 'निर्माण', '1732', 'पेशव', 'बाजीर', 'प्रथम', 'मुख्य', 'न

In [52]:
# ==========================================
# A. ENGLISH PROCESSING (NLTK)
# ==========================================
# Tokenize and clear punctuation via .isalnum()
en_total_tokens = [t for t in nltk.word_tokenize(documents["English"].lower()) if t.isalnum()]
en_filtered = [t for t in en_total_tokens if t not in en_stopwords]

# Lemmatization
lemmatizer = WordNetLemmatizer()
en_lemmas_list = [lemmatizer.lemmatize(t) for t in en_filtered]
en_clean_string = " ".join(en_lemmas_list)

# Stemming
en_stemmed_tokens = [porter_stemmer.stem(t) for t in en_filtered]
en_stemmed_clean_string = " ".join(en_stemmed_tokens)


# ==========================================
# B. HINDI PROCESSING (Custom Regex + Suffix Stripper)
# ==========================================
# Use 're' library to swap out Devanagari punctuation parameters
hi_text_clean = re.sub(r'[।,.!?\s]+', ' ', documents["Hindi"])
hi_total_tokens = [w for w in hi_text_clean.split(' ') if w]
hi_filtered = [w for w in hi_total_tokens if w not in hi_stopwords]

# Apply custom Hindi stemmer and capture frequency distribution logs
hi_stemmed_tokens = [hindi_stemmer(token) for token in hi_filtered]
hindi_counts = Counter(hi_stemmed_tokens)
hi_clean_string = " ".join(hi_stemmed_tokens)


# ==========================================
# C. MARATHI PROCESSING (Stanza Pipeline)
# ==========================================
mr_doc_initial = nlp_marathi(documents["Marathi"].strip())
# Extract total tokens excluding punctuation boundaries
mr_total_tokens = [w.text for s in mr_doc_initial.sentences for w in s.words if w.text not in ['.', ',', '।', '!', '?']]

# Map out lemmas and drop functional stopwords
mr_lemmas = [w.lemma for s in mr_doc_initial.sentences for w in s.words if w.text not in ['.', ',', '।', '!', '?']]
mr_filtered_lemmas = [L for L in mr_lemmas if L not in mr_stopwords and L.isalnum()]

# For Marathi, we consider Stanza's lemmatization (after filtering) as the stemming step.
mr_stemmed_tokens = mr_filtered_lemmas
mr_stemmed_clean_string = " ".join(mr_stemmed_tokens)

print("🔀 Multilingual token processing metrics calculated successfully!")

🔀 Multilingual token processing metrics calculated successfully!


In [53]:
print('Original English filtered tokens (before lemmatization):')
print(en_filtered)
print('\nLemmatized English tokens:')
print(en_lemmas_list)

print('\nCleaned English string:')
print(en_clean_string)


Original English filtered tokens (before lemmatization):
['shaniwar', 'wada', 'magnificent', 'historical', 'fortification', 'located', 'city', 'pune', 'constructed', '1732', 'peshwa', 'baji', 'rao', 'main', 'seat', 'peshwas', 'palace', 'became', 'political', 'heart', 'grand', 'capital', 'expanding', 'maratha', 'empire', 'architecture', 'displays', 'brilliant', 'blend', 'maratha', 'style', 'exquisite', 'mughal', 'design', 'elements', 'massive', 'foundations', 'consist', 'strong', 'stone', 'layers', 'upper', 'floors', 'built', 'timber', 'complex', 'feature', 'five', 'grand', 'gateways', 'including', 'famous', 'delhi', 'darwaza', 'facing', 'north', 'spectacular', 'structure', 'stood', 'proudly', 'within', 'inner', 'courtyard', 'walls', 'unfortunately', 'unexpected', 'devastating', 'fire', 'destroyed', 'entire', 'palace', 'complex', 'today', 'surviving', 'stone', 'ruins', 'attract', 'thousands', 'tourists', 'world', 'stands', 'timeless', 'symbol', 'maharashtra', 'rich', 'historical', 'heri

In [54]:
print('English filtered tokens (before stemming):')
print(en_filtered)
print('\nStemmed English tokens:')
print(en_stemmed_tokens)

print('\nCleaned English stemmed string:')
print(en_stemmed_clean_string)

English filtered tokens (before stemming):
['shaniwar', 'wada', 'magnificent', 'historical', 'fortification', 'located', 'city', 'pune', 'constructed', '1732', 'peshwa', 'baji', 'rao', 'main', 'seat', 'peshwas', 'palace', 'became', 'political', 'heart', 'grand', 'capital', 'expanding', 'maratha', 'empire', 'architecture', 'displays', 'brilliant', 'blend', 'maratha', 'style', 'exquisite', 'mughal', 'design', 'elements', 'massive', 'foundations', 'consist', 'strong', 'stone', 'layers', 'upper', 'floors', 'built', 'timber', 'complex', 'feature', 'five', 'grand', 'gateways', 'including', 'famous', 'delhi', 'darwaza', 'facing', 'north', 'spectacular', 'structure', 'stood', 'proudly', 'within', 'inner', 'courtyard', 'walls', 'unfortunately', 'unexpected', 'devastating', 'fire', 'destroyed', 'entire', 'palace', 'complex', 'today', 'surviving', 'stone', 'ruins', 'attract', 'thousands', 'tourists', 'world', 'stands', 'timeless', 'symbol', 'maharashtra', 'rich', 'historical', 'heritage', 'pride'

In [57]:
print('Original Marathi total tokens (after punctuation removal):')
print(mr_total_tokens)
print('\nMarathi lemmas (before stopword filtering):')
print(mr_lemmas)
print('\nMarathi filtered/stemmed lemmas:')
print(mr_filtered_lemmas)

print('\nCleaned Marathi string:')
print(mr_stemmed_clean_string)

Original Marathi total tokens (after punctuation removal):
['शनिवार', 'वाडा', 'हा', 'पुणे', 'शहरा', 'तील', 'एक', 'अत्यंत', 'भव्य', 'आणि', 'ऐतिहासिक', 'किल्ला', 'आहे', 'या', 'वास्तू', 'ची', 'उभारणी', '173', '2', 'मध्ये', 'पेशवे', 'बाजीराव', 'पहिले', 'यांनी', 'मुख्य', 'निवासस्थान', 'म्हणून', 'केली', 'हा', 'राजवाडा', 'बलाढ्य', 'मराठा', 'साम्रा', 'ज्याचे', 'प्रमुख', 'राजकीय', 'आणि', 'प्रशा', 'कीी', 'केंद्र', 'बनला', 'याच्या', 'वास्तुकले', 'ध्ये', 'मराठा', 'साम्राज्य', 'शैली', 'आणि', 'उत्कृष्ट', 'मुघल', 'डिझाइनचे', 'सुंदर', 'मिश्रण', 'दिसून', 'येते', 'किल्ला', 'चा', 'पाया', 'मजबूत', 'दगडा', 'चा', 'बनवला', 'असून', 'वरील', 'मजले', 'लाकडा', 'चे', 'बांधण्यात', 'आले', 'होते', 'या', 'ऐतिहासिक', 'वास्तूला', 'एकूण', 'पाच', 'मोठे', 'दरवाजे', 'असून', 'प्रसिद्ध', 'दिल्ली', 'दरवाजा', 'उत्तरे', 'कडे', 'तोंड', 'करून', 'आहे', 'एकदा', 'या', 'अंतर्गत', 'प्रांगणात', 'एक', 'भव्य', 'आणि', 'देखणी', 'सात', 'मजली', 'इमारत', 'उभी', 'होती', 'दुर्दैवाने', '1828', 'मध्ये', 'लागले', 'ल्या', 'एका', 'भीषण', 'आगीत', 'संप

In [56]:
print('--- Summary of Token Counts ---')
display(summary_df[['Language', 'Total Raw Tokens (Initial)', 'Processed Roots (Final)']])

--- Summary of Token Counts ---


,Language,Total Raw Tokens (Initial),Processed Roots (Final)
0,English (Lemmatized),137,88
1,Hindi (Custom Stemmed),146,89
2,Marathi (Lemmatized),139,11
